# Feature Demo: Create and Link Instance (API + CLI)

This notebook shows the same create workflow through both interfaces:

1. Create `cell-type` from datasheet
2. Create `cell-instance` linked to dataset
3. Compare API and CLI outputs


In [ ]:
from pathlib import Path
import json
import re
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src').exists(), f'Repo root not found from {Path.cwd()}'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from battinfo.api import create_cell_type_from_datasheet, create_cell_instance
from battinfo.cli import app
from typer.testing import CliRunner

runner = CliRunner()
tmp = ROOT / 'tmp_feature_create'
tmp.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)
print('tmp:', tmp)


## 1) Create via Python API

In [ ]:
datasheet = ROOT / 'assets' / 'examples' / 'cells' / 'A123__ANR26650M1-B.datasheet.json'
api_cell_type_path = tmp / 'api.cell-type.generated.json'
api_cell_instance_path = tmp / 'api.cell-instance.generated.json'

api_cell_type = create_cell_type_from_datasheet(
    datasheet,
    uid='7d9k2m4p8t3x6nq5',
    out_path=api_cell_type_path,
)
api_cell_instance = create_cell_instance(
    type_id=api_cell_type['cell_type']['id'],
    uid='3m6k9t2p7x4h9nq8',
    serial_number='LAB-API-001',
    dataset_id='https://w3id.org/battinfo/dataset/1f8r-6v2k-9p4m-3t7x',
    out_path=api_cell_instance_path,
)
print('API cell_type.id:', api_cell_type['cell_type']['id'])
print('API cell_instance.id:', api_cell_instance['cell_instance']['id'])


## 2) Create via CLI

In [ ]:
cli_cell_type_path = tmp / 'cli.cell-type.generated.json'
cli_cell_instance_path = tmp / 'cli.cell-instance.generated.json'

res = runner.invoke(
    app,
    [
        'create',
        'cell-type',
        '--datasheet',
        str(datasheet),
        '--uid',
        '8n2p1c4m7t5r6v3x',
        '--out',
        str(cli_cell_type_path),
        '--format',
        'json',
    ],
)
assert res.exit_code == 0, res.stdout
cli_cell_type_status = json.loads(res.stdout)

res = runner.invoke(
    app,
    [
        'create',
        'cell-instance',
        '--type-id',
        cli_cell_type_status['id'],
        '--uid',
        '5r4t3x2p9m8k1n7q',
        '--serial-number',
        'LAB-CLI-001',
        '--dataset-id',
        'https://w3id.org/battinfo/dataset/1f8r-6v2k-9p4m-3t7x',
        '--out',
        str(cli_cell_instance_path),
        '--format',
        'json',
    ],
)
assert res.exit_code == 0, res.stdout
cli_cell_instance_status = json.loads(res.stdout)

print('CLI cell_type.id:', cli_cell_type_status['id'])
print('CLI cell_instance.id:', cli_cell_instance_status['id'])


## 3) Parity and Policy Checks

In [ ]:
api_inst = json.loads(api_cell_instance_path.read_text(encoding='utf-8'))
cli_inst = json.loads(cli_cell_instance_path.read_text(encoding='utf-8'))

uid = r'[0-9a-hjkmnp-tv-z]{4}(?:-[0-9a-hjkmnp-tv-z]{4}){3}'
pat_cell = re.compile(rf'^https://w3id\.org/battinfo/cell/{uid}$')
pat_type = re.compile(rf'^https://w3id\.org/battinfo/cell-type/{uid}$')

print('API ids valid:', bool(pat_cell.fullmatch(api_inst['cell_instance']['id'])), bool(pat_type.fullmatch(api_inst['cell_instance']['type_id'])))
print('CLI ids valid:', bool(pat_cell.fullmatch(cli_inst['cell_instance']['id'])), bool(pat_type.fullmatch(cli_inst['cell_instance']['type_id'])))
print('API dataset link:', api_inst.get('provenance', {}).get('dataset_id'))
print('CLI dataset link:', cli_inst.get('provenance', {}).get('dataset_id'))


In [ ]:
# Cleanup (comment out if you want to inspect generated files)
for p in tmp.glob('*.json'):
    p.unlink()
tmp.rmdir()
print('cleaned')


## Debug tips

- Use fixed `--uid` values in notebooks for deterministic outputs.
- If create fails, inspect schema error text first; it usually points to the exact field.
- Keep serial numbers as metadata only, never as canonical identity.
